In [1]:
! pip install langchain_chroma langchain_community langchain_core langchain_openai vllm

In [2]:
from sys import get_coroutine_origin_tracking_depth
from typing_extensions import TypedDict, Annotated
from typing import List, Literal
from langchain_core.documents import Document
from operator import add
from pydantic import BaseModel, Field


class State(TypedDict):
  """
  state scheme for RAG
  contains the following
  question: question asked by the user
  generation: answer generated by LLM
  docs: documents as context
  retry_count: the count of loops happened to prevent infinite ReAct loops
  """

  question: str
  generation: str
  docs: Annotated[List[Document], add]
  raw_docs: List[Document]
  retry_count: int
  grade: Literal['good', 'bad']

class Grade(BaseModel):
    """Binary score for relevance check."""
    grade: Literal['good', 'bad'] = Field(
        description="Score 'good' if the answer resolves the question using the docs, else 'bad'."
    )

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

graph = StateGraph(state_schema=State)

vectorDB = Chroma(
      persist_directory='./db',
      embedding_function=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
  )

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="empty", # vLLM doesn't need a real key
    model="Qwen/Qwen2.5-3B-Instruct",
    temperature=0
)

llm_with_structured_output = llm.with_structured_output(Grade)

/tmp/ipykernel_42573/4005325735.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [4]:
def rewrite_query(state: State):
  # we take state['question'] and state['docs']
  # we format them into a prompt
  # we pass them into LLM via local VLLM
  # we update the state['question']

  query = state['question']
  doc_string = '\n'.join([doc.page_content for doc in state['docs']])

  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are an expert query optimizer. Your task is to rewrite the user's question to be more effective for document retrieval. Analyze the provided context documents to identify key terms, concepts, or potential ambiguities related to the original question. Generate a single, concise, and optimized query that would yield more relevant results from a vector database. If the original question is already good, you can return it as is. Do not answer the question, only rewrite it."),
      ("human", "Context: {context}\n\nQuestion: {question}")
  ])

  chain = prompt | llm | StrOutputParser()

  optimized_question_string =  chain.invoke({
    'context': doc_string,
    'question': query
  })

  return {'question': optimized_question_string}

In [5]:
class QueryExpansion(BaseModel):
    """Expanded search queries for retrieval."""
    queries: List[str] = Field(description="3-5 diverse search variations of the original question")
    hypothetical_answer: str = Field(description="A 2-sentence hypothetical answer to the question")

def query_generator(query, k):
  # query the users question
  # generate many k queries from the original query
  # return k queries

  prompt = ChatPromptTemplate.from_messages([
      ("system", f"You are a helpful assistant that generates {k} diverse search queries based on a given user question."),
      ("human", "Generate diverse search queries for the following question: {question}")
  ])

  llm_with_structured_output = llm.with_structured_output(QueryExpansion)

  chain = prompt | llm_with_structured_output

  qe_output = chain.invoke({'question': query})

  generated_queries = qe_output.queries
  hypothetical_answer = qe_output.hypothetical_answer

  all_search_terms = [query] + [q.strip() for q in generated_queries if q.strip()][:k] + [hypothetical_answer]

  return all_search_terms

In [6]:
def retrieve(state: State):
  # generate 3 queries based on the original query and a hypothetical answer to help us in retrieving process
  # query the users question
  # retrieve the answers according to the db query

  generated_queries = query_generator(state['question'], 3)

  all_queried_docs = []
  for query_text in generated_queries:
    all_queried_docs.extend(vectorDB.similarity_search(query_text, k=10))

  unique_docs = []
  seen_page_contents = set()
  for doc in all_queried_docs:
    if doc.page_content not in seen_page_contents:
      unique_docs.append(doc)
      seen_page_contents.add(doc.page_content)

  return {'raw_docs': unique_docs}

In [7]:
def rerank(state: State):
  # pairs the new docs and question
  # rerank them by score using cross encoder
  # retrive the top k
  k = 3

  pairs = [[state['question'], doc.page_content] for doc in state['raw_docs']]
  scores = cross_encoder.predict(pairs)

  reranked_docs = [x for _, x in sorted(zip(scores, state['raw_docs']), key=lambda pair: pair[0], reverse=True)]

  return {'docs': reranked_docs[:k]}


In [9]:
def generate(state: State):
  # we take state['question'] and state['docs']
  # we format them into a prompt
  # we pass them into LLM via local VLLM
  # we update the state['generation']

  # =================================
  # for i, doc in enumerate(state['docs']):
  #     print(f"Chunk {i+1}: {doc.page_content[:200]}...") # recapturing how the chunks passed
  # print("----------------------------\n")
  # =================================
  doc_string = '\n'.join([doc.page_content for doc in state['docs']])

  prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a precise, no-nonsense customer support agent.
    You must answer the user's question using ONLY the provided context.
    If the context does not contain the answer, you must state that you do not have enough information.
    Do not hallucinate or rely on outside knowledge."""),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

  chain = prompt | llm | StrOutputParser()

  generation =  chain.invoke({
    'context': doc_string,
    'question': state['question']
  })

  return {'generation': generation}

In [10]:
def reflect(state: State):
  doc_string = '\n'.join([doc.page_content for doc in state['docs']])

  reflect_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a strict QA grader evaluating an AI support agent.
    Your job is to determine if the 'Generated Answer' successfully resolves the 'Question' AND is completely grounded in the provided 'Context'.
    You are not grading formatting; you are grading factual accuracy against the context.
    Output 'good' if it is accurate and resolves the query. Output 'bad' if it hallucinates or fails to answer."""),
    ("human", "Question: {question}\n\nContext: {context}\n\nGenerated Answer: {generation}")
])

  chain = reflect_prompt | llm_with_structured_output

  grade =  chain.invoke({
    'context': doc_string,
    'question': state['question'],
    'generation': state['generation']
  })

  return {'grade': grade.grade, 'retry_count': state['retry_count'] + 1}

In [11]:
def route(state: State):
  if state['retry_count'] >= 3 or state['grade'] == 'good':
    return END

  if state['grade'] == 'bad':
    return 'rewrite'

In [12]:
graph.add_node('rewrite', rewrite_query)
graph.add_node('retrieve', retrieve)
graph.add_node('rerank', rerank)
graph.add_node('generate', generate)
graph.add_node('reflect', reflect)

graph.add_edge(START, 'retrieve')
graph.add_edge('retrieve', 'rerank')
graph.add_edge('rerank', 'generate')
graph.add_edge('generate', 'reflect')
graph.add_conditional_edges('reflect', route, {END: END, 'rewrite': 'rewrite'})
graph.add_edge('rewrite', 'retrieve')

In [13]:
!pip install beautifulsoup4 requests

In [14]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
url = "https://en.wikipedia.org/wiki/ChatGPT"
response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")

content_div = soup.find("div", {"id": "mw-content-text"})

if not content_div:
    raise ValueError(f"Failed to find main content div for URL: {url}")

for sup in content_div.find_all("sup", class_="reference"):
    sup.decompose()
for reflist in content_div.find_all(class_="reflist"):
    reflist.decompose()

paragraphs = content_div.find_all("p")
clean_text = "\n\n".join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])

doc = Document(page_content=clean_text, metadata={"source": url})

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " "]
)

splitted_docs = text_splitter.split_documents([doc])

print(f"Total chunks generated: {len(splitted_docs)}\n")
for i, chunk in enumerate(splitted_docs[:3]):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("-" * 20 + "\n")

vectorDB.add_documents(splitted_docs)

Total chunks generated: 46

--- Chunk 1 ---
ChatGPT is a generative artificial intelligence chatbot developed by OpenAI. It was released in November 2022. It uses large language models—specifically generative pre-trained transformers (GPTs)—to generate text, speech, and images in response to user prompts. It is credited with accelerating the AI boom, an ongoing period marked by rapid investment and public attention toward the field of artificial intelligence (AI). OpenAI operates the service on a freemium model. Users can interact with ChatGPT through text, audio, and image prompts.

ChatGPT was quickly adopted, reaching 100 million monthly active users two months after its release and 900 million weekly active users in February 2026. It has been lauded for its potential to transform numerous professional fields, and has instigated public debate about the nature of creativity and the future of knowledge work.
--------------------

--- Chunk 2 ---
The chatbot has also been criticized fo

['1e793148-ad49-491e-a7a4-6209948333f7',
 '87812fb6-7d57-44af-8487-87c986e460e6',
 '95035280-8fa5-4858-a9c8-14b1f4865b3b',
 '02867608-21f0-400f-9e28-181a8d4c5d3c',
 'a071d08e-ceed-4e9c-919e-08dba727de29',
 '52184e8d-92f1-4356-9f13-f7b2ea1bf681',
 'f0a64220-7bf6-446c-840d-724573a7d82e',
 'f05826ba-e789-4f75-a21f-7bda72d1a660',
 '7c94fdf0-df13-4b01-8fab-fccfe40de801',
 '739ec1ef-561a-4b7e-a0a0-185638a4549d',
 'c959690b-ba60-4cd9-8972-1525529f3b3d',
 'd5b20868-ac73-437b-a2de-c13ddd58adac',
 '812bf697-3f52-481c-b3a6-2bd3a012d44d',
 'cd917433-fcbe-4846-a4da-059d570d41a8',
 'c1013b3e-41c1-4915-8194-d9cf9f301b78',
 '6df069a0-008a-4fe4-9591-d97752311242',
 '15587e28-1b43-42bd-9d59-79c8a35f26bf',
 '4c2dfa0e-87e3-43c5-aace-ee6d15d08663',
 'c1a0bee0-f03d-4ee0-9a98-08eb59a70a71',
 'cb5091b3-44a0-4011-a90f-0badba84ec5e',
 '278958b2-3eec-48db-81ca-3e27855d0821',
 '8716da1f-89c2-4d57-afa2-b7d6708e1ade',
 '0f0f4246-7e67-473f-b1ec-5bdd3eba119b',
 'fee4c56d-6fa9-46a6-bcd3-8260e7999583',
 '75d5e3f0-bb34-

In [15]:
import subprocess
import time

subprocess.run(["pkill", "-f", "vllm"])
time.sleep(3)

server_cmd = "python -u -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-3B-Instruct --gpu-memory-utilization 0.7 --max-model-len 4096 --enforce-eager"

print("Booting Qwen 3B server...")
with open("server.log", "w") as log_file:
    server_process = subprocess.Popen(
        server_cmd.split(),
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

print("Server process spawned with PID:", server_process.pid)

Booting Qwen 3B server...
Server process spawned with PID: 43935


In [18]:
!cat server.log

(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299] 
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.19.1
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-3B-Instruct
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:299] 
(APIServer pid=43935) INFO 04-18 22:44:18 [utils.py:233] non-default args: {'model': 'Qwen/Qwen2.5-3B-Instruct', 'max_model_len': 4096, 'enforce_eager': True, 'gpu_memory_utilization': 0.7}
(APIServer pid=43935) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=43935) INFO 04-18 22:44:42 [model.py:549] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=43935) WARNING 04-18 22:44:42 [model

In [19]:
app = graph.compile()

initial_state = {
    "question": "Who developed ChatGPT and when was it launched?",
    "docs": [],
    "raw_docs": [],
    "retry_count": 0,
    "generation": "",
    "grade": "bad"
}

final_state = app.invoke(initial_state)

print("Final Answer:", final_state['generation'])
print("Total Retries:", final_state['retry_count'])


Final Answer: ChatGPT was developed by OpenAI and was launched in November 2022.
Total Retries: 1
